# 04 — EDA I: Análise Exploratória Geral

**Objetivo:** compreender a população e a variável resposta antes de qualquer modelagem.

**Roteiro:**
1. Volume de clientes e snapshots
2. Distribuição do target
3. Evolução temporal de clientes e eventos
4. Perfil de consumo (produtos, formas de pagamento, valores)
5. Comparação inicial entre quem recompra e quem não recompra

In [0]:
# ── Configuração central (idem notebook 00) ──
SOURCE  = "samples.bakehouse.sales_transactions"
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"

from pyspark.sql import functions as F

population = spark.table(f"{CATALOG}.{SCHEMA}.population_active")
features   = spark.table(f"{CATALOG}.{SCHEMA}.customer_features")
target_tb  = spark.table(f"{CATALOG}.{SCHEMA}.population_target")
trans      = spark.table(SOURCE).drop("cardNumber").withColumn("tx_date", F.to_date("dateTime"))

df_model = (
    features
    .join(target_tb, on=["reference_date", "customerID"], how="inner")
)
print("Base modelo (features + target):", df_model.count(), "linhas")

In [0]:
# 1) Volume por snapshot e distribuição do target
display(
    df_model.groupBy("reference_date")
    .agg(
        F.count("*").alias("clientes"),
        F.sum("target").alias("recompras"),
        F.round(F.avg("target") * 100, 2).alias("taxa_recompra_%"),
    )
    .orderBy("reference_date")
)

# Distribuição global
display(
    df_model.groupBy("target")
    .agg(F.count("*").alias("linhas"), F.round(F.count("*") * 100 / df_model.count(), 2).alias("%"))
)

In [0]:
# 2) Evolução temporal de transações e clientes ativos (base original)
display(
    trans.groupBy("tx_date")
    .agg(
        F.count("*").alias("transacoes"),
        F.countDistinct("customerID").alias("clientes_ativos"),
        F.round(F.sum("totalPrice"), 2).alias("receita_dia"),
    )
    .orderBy("tx_date")
)

In [0]:
# 3) Perfil de consumo: produtos e formas de pagamento
display(
    trans.groupBy("product")
    .agg(
        F.count("*").alias("transacoes"),
        F.sum("quantity").alias("unidades"),
        F.round(F.sum("totalPrice"), 2).alias("receita"),
    )
    .orderBy(F.desc("receita"))
)

display(
    trans.groupBy("paymentMethod")
    .agg(F.count("*").alias("transacoes"), F.round(F.avg("totalPrice"), 2).alias("ticket_medio"))
    .orderBy(F.desc("transacoes"))
)

# Distribuição de valores transacionados
display(trans.select("quantity", "unitPrice", "totalPrice").summary())

In [0]:
# 4) Comparação inicial: eventos vs. não eventos nas features comportamentais
feat_num = [
    "recencia_dias", "freq_compras_5d", "valor_total_5d", "ticket_medio_5d",
    "qtde_itens_5d", "produtos_distintos_5d", "franquias_distintas_5d",
    "preco_medio_5d", "tendencia_2d_vs_3a5",
]

display(
    df_model.groupBy("target")
    .agg(*[F.round(F.avg(c), 3).alias(c) for c in feat_num])
    .orderBy("target")
)

# Taxa de recompra por recência (proxy do poder preditivo)
display(
    df_model.groupBy("recencia_dias")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("target"), 4).alias("taxa_recompra"),
    )
    .orderBy("recencia_dias")
)

## Conclusões

* Confirme: população estável entre snapshots e taxa de recompra na faixa de 38–45%.
* Compare a taxa de recompra por `recencia_dias` — é esperado que quanto maior a recência, menor a recompra (principal sinal preditivo em modelos de propensão).
* Se eventos e não eventos parecerem idênticos em **todas** as dimensões, o dataset sintético pode ter pouca relação sinal→target — o AUC então ficará perto de 0,5. Isso é um resultado válido do estudo e deve ser reportado com honestidade.
* **Próximo passo:** `05_eda_ii` — qualidade e poder preditivo formal das variáveis (IV, PSI, correlações).